# Notebook 00: The Python You'll Need

A gentle, **runnable** companion to [`docs/week_00_python_prerequisites.md`](../docs/week_00_python_prerequisites.md).

This notebook teaches the Python you need for the course **by pointing at this project's own code**, so nothing feels abstract. You only need basic Python (variables, functions, loops, `if`) and basic data structures (lists, dicts) to follow along.

> **How to use it:** read once, run the cells, then *revisit a section whenever a later week uses a symbol you don't recognize.*

**Before running:** make sure you installed the package with `pip install -e .` from the repo root (see Week 1). Otherwise the imports below fail with `ModuleNotFoundError`.

## Setup and Imports

We import the real building blocks of this project and use them throughout.

In [ ]:
from pathfinding_lab.core.grid import Grid
from pathfinding_lab.core.node import Node
from pathfinding_lab.core.result import SearchResult
from pathfinding_lab.core.types import Position, CellType, MovementMode

print("Imports OK — package is installed correctly.")

## 1. Tuples and Tuple Unpacking

A **tuple** is an ordered, fixed collection: `(2, 5)`. This project represents every grid cell as a `(row, col)` tuple aliased as `Position`. **Tuple unpacking** splits it into named variables.

In [ ]:
position = (2, 5)
row, col = position          # tuple unpacking
print("row =", row, "| col =", col)

# Tuples can be dict keys and set members (lists cannot):
costs = {(0, 0): 0.0, (2, 5): 7.0}
print("cost at (2,5):", costs[(2, 5)])

**Why we use this here:** coordinates never change once created, so a tuple is the natural choice — and only tuples can be dict keys / set members, which the algorithms rely on.

## 2. Sets vs Lists vs Dicts — and Big-O in Plain Words

**Big-O** describes how work grows as data grows. `pos in a_set` is **O(1)** (constant, fast) while `pos in a_list` is **O(n)** (scans every item). That is why obstacles live in a **set**.

In [ ]:
grid = Grid(width=10, height=10, obstacle_density=0.2, random_seed=42)
grid.generate_obstacles(start=(0, 0), goal=(9, 9))

print("obstacles is a:", type(grid.obstacles).__name__)
some_cell = next(iter(grid.obstacles))
print("Is", some_cell, "an obstacle? ->", grid.is_obstacle(some_cell), "(an O(1) set lookup)")

In [ ]:
import timeit

big_set = set(range(100_000))
big_list = list(range(100_000))
target = 99_999  # worst case for the list (last item)

set_time = timeit.timeit(lambda: target in big_set, number=10_000)
list_time = timeit.timeit(lambda: target in big_list, number=10_000)
print(f"set  lookup (O(1)): {set_time:.4f}s")
print(f"list lookup (O(n)): {list_time:.4f}s")
print("The list is dramatically slower because it may scan every item.")

**Why we use this here:** algorithms ask "is this blocked?" / "visited yet?" constantly. Sets and dicts answer in O(1); lists would be O(n) and make search slow. Later weeks reuse the words *O(1)* and *O(n)* — they mean exactly this.

## 3. Classes and `self` (Just Enough to Read `Grid`)

A **class** bundles data with the functions (**methods**) that use it. `self` means "this instance." `Grid` stores its size and obstacles and exposes methods like `is_valid` and `get_neighbors`.

In [ ]:
g = Grid(width=5, height=4)          # __init__ runs here
print("width:", g.width, "| height:", g.height)   # data stored on the instance
print("is (3, 2) valid?", g.is_valid((3, 2)))     # a method call; Python passes `self`
print("is (9, 9) valid?", g.is_valid((9, 9)))

**Why we use this here:** the grid owns lots of related state and behavior; a class keeps them together so every algorithm shares one consistent `Grid`.

## 4. Dataclasses (`@dataclass`, default fields)

A **dataclass** writes the boilerplate constructor for you. `Node` and `SearchResult` are dataclasses with sensible default field values.

In [ ]:
n = Node(position=(2, 5))
print("position:", n.position)
print("g_cost default:", n.g_cost, "(float('inf') = 'not reached yet')")
print("is_obstacle default:", n.is_obstacle)

r = SearchResult(algorithm_name="Demo", success=False)
print("path default:", r.path, "(an empty list, via field(default_factory=list))")

**Why we use this here:** `Node`/`SearchResult` are pure data holders. Dataclasses give a clean constructor, defaults, and a readable display for free. Mutable defaults like `path` must use `field(default_factory=list)` so instances don't share one list.

## 5. Dunder Methods (`__lt__`, `__eq__`, `__hash__`)

"Dunder" = double underscore. They hook your objects into Python operators and containers: `__eq__` -> `==`, `__hash__` -> use in a set, `__lt__` -> `<` (what a priority queue needs).

In [ ]:
a = Node(position=(1, 1))
b = Node(position=(1, 1))   # same cell
c = Node(position=(9, 9))

print("a == b? (uses __eq__):", a == b)          # equal because same position
print("a in {b, c}? (uses __hash__):", a in {b, c})

a.f_cost = 3.0
c.f_cost = 8.0
print("a < c? (uses __lt__, compares f_cost):", a < c)

import heapq
pq = []
heapq.heappush(pq, a)
heapq.heappush(pq, c)
print("priority queue pops cheapest first:", heapq.heappop(pq).f_cost)

**Why a priority queue needs ordering:** to return the smallest item it evaluates `a < b` (`__lt__`). Without it, Python raises `TypeError: '<' not supported`. (The real algorithms push plain `(cost, position)` tuples, which Python already orders — `Node` shows the idea explicitly.)

## 6. Enums (`MovementMode`) — Why Not Just Strings

An **enum** is a fixed set of named choices. `MovementMode` has exactly two members, so typos like `"8-dir"` are impossible.

In [ ]:
print("Members:", list(MovementMode))

g4 = Grid(8, 8, movement_mode=MovementMode.FOUR_DIRECTIONAL)
g8 = Grid(8, 8, movement_mode=MovementMode.EIGHT_DIRECTIONAL)
print("4-dir neighbors of (3,3):", len(g4.get_neighbors((3, 3))))
print("8-dir neighbors of (3,3):", len(g8.get_neighbors((3, 3))))

**Why we use this here:** there are exactly two movement modes; an enum makes valid options explicit and typo-proof. CellType is another enum (EMPTY/OBSTACLE/START/GOAL/...).

## 7. Type Hints — How to *Read* Them

Type hints document what a value should be. You mostly need to **read** them. Look at A*'s signature: `heuristic` is typed `Callable[[Position, Position], float]` — i.e. *a function* taking two positions and returning a number.

In [ ]:
import inspect
from pathfinding_lab.algorithms.astar import astar
print(inspect.signature(astar))

# `heuristic` is a FUNCTION, so we pass a function (not a number):
from pathfinding_lab.heuristics.manhattan import manhattan_distance
print("manhattan_distance((0,0),(3,4)) =", manhattan_distance((0, 0), (3, 4)))

**Why we use this here:** the signature becomes documentation — `Callable[[Position, Position], float]` instantly says "pass a heuristic *function*," no digging required.

## 8. `collections.deque`, `heapq`, and `float('inf')`

The data structure must match the search order: FIFO -> `deque` (BFS), cheapest-first -> `heapq` (Dijkstra/A*), "no cost known yet" -> `float('inf')`.

In [ ]:
from collections import deque
frontier = deque([(0, 0)])
frontier.append((0, 1))         # add to back  (O(1))
print("popleft (FIFO):", frontier.popleft())   # remove from front (O(1))

import heapq
pq = [(5.0, (1, 1))]
heapq.heappush(pq, (2.0, (0, 1)))   # cheaper option
print("heappop (cheapest first):", heapq.heappop(pq))

best = float('inf')
print("any real cost < inf?", 7.0 < best)   # the first real path always wins

**Why we use this here:** FIFO search needs O(1) front-removal (`deque`); cheapest-first search needs an efficient min (`heapq`); `float('inf')` is the "not reached yet" starting cost so the first real path replaces it.

## 9. `try/except` Basics

`try/except` attempts something risky and handles failure instead of crashing — used in the Week 1 exercise and the Gradio UI input validation.

In [ ]:
def parse_coordinate(text):
    try:
        row, col = text.split(",")
        return (int(row), int(col))
    except ValueError:
        return "Invalid coordinate — expected 'row,col' with whole numbers."

print(parse_coordinate("2,5"))
print(parse_coordinate("two,five"))

**Why we use this here:** a typo becomes a friendly message rather than a stack trace, keeping exercises and the app robust.

## You're Ready

You can now **read** every algorithm file in this repo. Keep this notebook (and [`docs/week_00_python_prerequisites.md`](../docs/week_00_python_prerequisites.md)) open and revisit a section whenever a later week uses a symbol you don't recognize.

**Next:** [Week 1 — Python Project Setup](../docs/week_01_python_project_setup.md).